In [ ]:
import os
import gc
import pickle
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ All libraries imported successfully.")

In [ ]:
# ── UPDATE THIS PATH ─────────────────────────────────────────────────────────
dataset_path = r"D:\VS Code\Python\plant disease idetification\plantvillage dataset\color"
# ─────────────────────────────────────────────────────────────────────────────

IMAGE_SIZE  = 32
BATCH_SIZE  = 32
SAMPLE_SIZE = 5000   # sufficient for SVM/RF, keeps RAM low

# Separate generator — no augmentation, no validation_split
datagen_flat = ImageDataGenerator(rescale=1./255)
flat_gen = datagen_flat.flow_from_directory(
    dataset_path,
    target_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'sparse',    # integer labels
    shuffle     = True,
    seed        = 42
)

# Collect SAMPLE_SIZE images batch-by-batch
sample_x, sample_y = [], []
for imgs, lbls in flat_gen:
    sample_x.append(imgs)
    sample_y.append(lbls)
    if sum(len(b) for b in sample_x) >= SAMPLE_SIZE:
        break

sample_x = np.concatenate(sample_x, axis=0)[:SAMPLE_SIZE]
sample_y = np.concatenate(sample_y, axis=0)[:SAMPLE_SIZE].astype(int)

# Flatten → PCA
X_flat = sample_x.reshape(len(sample_x), -1).astype(np.float32)
del sample_x; gc.collect()

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_flat, sample_y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = sample_y
)
del X_flat; gc.collect()

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
del X_train_s, X_test_s; gc.collect()

print(f"✅ Feature Processing complete.")
print(f"   Sample drawn       : {SAMPLE_SIZE} images")
print(f"   PCA components     : 100")
print(f"   Train PCA shape    : {X_train_pca.shape}")
print(f"   Test  PCA shape    : {X_test_pca.shape}")
print(f"   Variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%")